####  Inserindo genes que estão presentes no fujit e ausentes no mathys
A ideia aqui é inserir colunas nos s genes que estão presentes no fujita mais aussnetes no mathys, dentro do arquivo.csv da matriz de expresão do matis com esses genes zerados ara que possa ser feita a previsão dos dados faltants utilizando o hopifield.
 temos alguns problemas para resolver:
1. Como inserir as colunas no arquivo.csv da matriz de expresão?
2. existem genes que tem o codigo de gene igual nos dois arquivos mais o nome do gene em cad matiz é diferente para o mesmo codigom isso complica um pouquinho ao nosso trabalho poispara utilizar o hopifield precisamos dos mesmos nomes de genes ou codigos na mesma o parrdem de colunas para que a previsão possa ser feita. (HTML <span> (Recomendado): <span style="color:blue">Precisos resolver como fazer isso?🤔</span>)

In [39]:
import scanpy as sc
import pandas as pd

In [40]:
import gc

PATH_F = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Controle_qualidade/dataF/MatrizfiltradaenormalizadaF/matrizFiltradaeNormalizadaF.h5ad"
PATH_M = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Controle_qualidade/dataM/matrizFiltradaeNormalizadaM.h5ad"

# backed='r' → só carrega metadados; a matriz X permanece no disco
adataf = sc.read_h5ad(PATH_F, backed='r')
print(f"Fujita (backed): {adataf.shape}")

Fujita (backed): (40913, 36601)


In [41]:
adatam = sc.read_h5ad(PATH_M, backed='r')
print(f"Mathys (backed): {adatam.shape}")

Mathys (backed): (47523, 32738)


---
## Passo 1 — Mapear gene_name → Ensembl ID em ambos os datasets

O problema: um mesmo gene pode ter nomes diferentes no Fujita e no Mathys
(ex: `AL627309.1` no Fujita ↔ `RP11-34P13.7` no Mathys).
A solução: substituir os nomes pelo **código Ensembl** (ENSG...) que é único e universal,
depois alinhar as colunas de ambos os AnnData pela mesma lista ordenada de IDs.

In [42]:
import numpy as np
import scipy.sparse as sp

PATH_FEATURES_F = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Controle_qualidade/dataF/features.tsvF.gz"
PATH_FEATURES_M = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Controle_qualidade/dataM/featuresM.tsv.gz"

def ler_features(path):
    """
    Lê um features.tsv(.gz) do 10x Genomics.
    Colunas: ensembl_id | gene_name | (tipo — opcional)
    Retorna dict {gene_name: ensembl_id}.
    """
    df = pd.read_csv(path, sep='\t', header=None, usecols=[0, 1],
                     names=['ensembl_id', 'gene_name'])
    # em caso de duplicata de nome, mantém a primeira ocorrência
    df = df.drop_duplicates(subset='gene_name', keep='first')
    return dict(zip(df['gene_name'], df['ensembl_id']))

In [43]:
map_f = ler_features(PATH_FEATURES_F)
map_m = ler_features(PATH_FEATURES_M)

print(f"Fujita  — total genes: {adataf.n_vars}, com Ensembl ID: {len(map_f)}")
print(f"Mathys  — total genes: {adatam.n_vars}, com Ensembl ID: {len(map_m)}")

# guarda a ordem original dos var_names antes de fechar os arquivos backed
var_names_f_original = adataf.var_names.tolist()

# fecha os file handles backed — X nunca chegou a ocupar RAM
adataf.file.close()
adatam.file.close()
del adataf, adatam
gc.collect()
print("Objetos backed liberados da memória.")

Fujita  — total genes: 36601, com Ensembl ID: 36591
Mathys  — total genes: 32738, com Ensembl ID: 32643
Objetos backed liberados da memória.


## Passo 2 — Renomear var_names para Ensembl ID e checar sobreposição

In [44]:
# Os conjuntos de Ensembl IDs são derivados diretamente dos mapas — sem carregar X
ids_f = set(map_f.values())
ids_m = set(map_m.values())
ids_comuns = ids_f & ids_m
ids_so_f   = ids_f - ids_m
ids_so_m   = ids_m - ids_f
ids_uniao  = ids_f | ids_m

print(f"Fujita  — genes com Ensembl ID: {len(ids_f)}")
print(f"Mathys  — genes com Ensembl ID: {len(ids_m)}")
print(f"Em comum:    {len(ids_comuns)}")
print(f"Só Fujita:   {len(ids_so_f)}")
print(f"Só Mathys:   {len(ids_so_m)}  ← serão excluídos")

# ordem de referência: TODOS os genes do Fujita na ordem original
# usa Ensembl ID onde disponível; mantém nome original como fallback
seen = set()
genes_ordenados = []
for gene_name in var_names_f_original:
    eid = map_f.get(gene_name, gene_name)  # fallback = nome original do gene
    if eid not in seen:
        genes_ordenados.append(eid)
        seen.add(eid)

n_com_ensembl = sum(1 for g in var_names_f_original if g in map_f)
n_sem_ensembl = len(var_names_f_original) - n_com_ensembl
print(f"\nEspaço gênico final: {len(genes_ordenados)} genes")
print(f"  {n_com_ensembl} com Ensembl ID  |  {n_sem_ensembl} mantidos com nome original")

Fujita  — genes com Ensembl ID: 36591
Mathys  — genes com Ensembl ID: 32643
Em comum:    30312
Só Fujita:   6279
Só Mathys:   2331  ← serão excluídos

Espaço gênico final: 36601 genes
  36591 com Ensembl ID  |  10 mantidos com nome original


## Passo 3 — Alinhar ambos os AnnData à mesma ordem de genes (união, zeros para ausentes)

In [45]:
import os
import anndata as ad

OUT_DIR = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Testes Hopifild"

gene_alvo_idx = {g: i for i, g in enumerate(genes_ordenados)}


def alinhar_direto(adata, ensembl_map, gene_alvo_idx, genes_ordenados, fill_value=0.0):
    """
    Rename + alinhamento em um único passo, sem cópias intermediárias.
    Usa Ensembl ID onde disponível; fallback = nome original do gene.
    Genes ausentes no adata recebem expressão `fill_value` (padrão 0.0).
    """
    n_celulas = adata.n_obs
    n_genes   = len(genes_ordenados)

    old_col_map      = np.full(adata.n_vars, -1, dtype=np.intp)
    present_new_cols = set()
    for old_i, gene_name in enumerate(adata.var_names):
        eid = ensembl_map.get(gene_name, gene_name)
        if eid in gene_alvo_idx:
            new_col = gene_alvo_idx[eid]
            old_col_map[old_i] = new_col
            present_new_cols.add(new_col)

    X_coo        = adata.X.tocoo() if sp.issparse(adata.X) else sp.coo_matrix(adata.X)
    new_cols_arr = old_col_map[X_coo.col]
    mask         = new_cols_arr >= 0

    X_novo = sp.csr_matrix(
        (X_coo.data[mask].astype(np.float32),
         (X_coo.row[mask], new_cols_arr[mask])),
        shape=(n_celulas, n_genes),
        dtype=np.float32,
    )
    del X_coo, new_cols_arr, mask

    if fill_value != 0.0:
        missing_cols = np.array(
            sorted(set(range(n_genes)) - present_new_cols), dtype=np.intp
        )
        if len(missing_cols) > 0:
            print(f"  Preenchendo {len(missing_cols)} colunas ausentes com {fill_value}...")
            n_fill       = len(missing_cols) * n_celulas
            fill_rows    = np.tile(np.arange(n_celulas, dtype=np.int32), len(missing_cols))
            fill_cols_arr = np.repeat(missing_cols.astype(np.int32), n_celulas)
            fill_data    = np.full(n_fill, fill_value, dtype=np.float32)
            X_fill = sp.csr_matrix(
                (fill_data, (fill_rows, fill_cols_arr)),
                shape=(n_celulas, n_genes),
                dtype=np.float32,
            )
            del fill_rows, fill_cols_arr, fill_data
            X_novo = X_novo + X_fill
            del X_fill
            gc.collect()
            print(f"  Preenchimento concluído.")

    var_novo = pd.DataFrame(index=pd.Index(genes_ordenados, name='ensembl_id'))
    return ad.AnnData(X=X_novo, obs=adata.obs.copy(), var=var_novo)


# ── Fujita ────────────────────────────────────────────────────────────────────
print("Carregando Fujita completo...")
adataf = sc.read_h5ad(PATH_F)
print(f"  shape original: {adataf.shape}")

print("Alinhando Fujita (fill=0.0 — sem alteração)...")
adataf_alinhado = alinhar_direto(adataf, map_f, gene_alvo_idx, genes_ordenados, fill_value=0.0)
del adataf
gc.collect()

out_f = os.path.join(OUT_DIR, "adataF_ensembl_alinhado.h5ad")
adataf_alinhado.write_h5ad(out_f)
print(f"  shape final: {adataf_alinhado.shape}")
del adataf_alinhado
gc.collect()
print(f"  salvo em {out_f}  ✓\n")

# ── Mathys ────────────────────────────────────────────────────────────────────
print("Carregando Mathys completo...")
adatam = sc.read_h5ad(PATH_M)
print(f"  shape original: {adatam.shape}")

print("Alinhando Mathys (genes ausentes → 0.5)...")
adatam_alinhado = alinhar_direto(adatam, map_m, gene_alvo_idx, genes_ordenados, fill_value=0.5)
del adatam
gc.collect()

out_m = os.path.join(OUT_DIR, "adataM_ensembl_alinhado.h5ad")
adatam_alinhado.write_h5ad(out_m)
print(f"  shape final: {adatam_alinhado.shape}")
del adatam_alinhado
gc.collect()
print(f"  salvo em {out_m}  ✓")

print("\nConcluído. Ambos os datasets alinhados e salvos.")

Carregando Fujita completo...
  shape original: (40913, 36601)
Alinhando Fujita (fill=0.0 — sem alteração)...
  shape final: (40913, 36601)
  salvo em /home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Testes Hopifild/adataF_ensembl_alinhado.h5ad  ✓

Carregando Mathys completo...
  shape original: (47523, 32738)
Alinhando Mathys (genes ausentes → 0.5)...
  Preenchendo 6289 colunas ausentes com 0.5...
  Preenchimento concluído.
  shape final: (47523, 36601)
  salvo em /home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Testes Hopifild/adataM_ensembl_alinhado.h5ad  ✓

Concluído. Ambos os datasets alinhados e salvos.


## Arquivo de rastreio — genes adicionados ao Mathys com valor 0.5

Lista todas as colunas que foram inseridas no Mathys (genes exclusivos do Fujita),
indicando o nome do gene, o Ensembl ID e a posição exata da coluna no espaço gênico alinhado.

In [46]:
# Mapa inverso: ensembl_id → gene_name original do Fujita
inv_map_f = {v: k for k, v in map_f.items()}

# ids_so_f = genes presentes APENAS no Fujita (definido no Passo 2)
tracking_rows = []
for eid in sorted(ids_so_f):
    if eid in gene_alvo_idx:
        tracking_rows.append({
            'gene_name'     : inv_map_f.get(eid, eid),
            'ensembl_id'    : eid,
            'posicao_coluna': gene_alvo_idx[eid],
            'valor_inserido': 0.5,
            'presente_fujita': True,
            'presente_mathys': False,
        })

df_tracking = (
    pd.DataFrame(tracking_rows)
    .sort_values('posicao_coluna')
    .reset_index(drop=True)
)

print(f"Total de colunas adicionadas ao Mathys com 0.5: {len(df_tracking)}")
print(f"\nPrimeiras 10 entradas do rastreio:")
print(df_tracking.head(10).to_string(index=False))

out_tracking = os.path.join(OUT_DIR, "tracking_genes_adicionados_mathys.csv")
df_tracking.to_csv(out_tracking, index=False)
print(f"\nArquivo de rastreio salvo em:\n  {out_tracking}")

Total de colunas adicionadas ao Mathys com 0.5: 6279

Primeiras 10 entradas do rastreio:
 gene_name      ensembl_id  posicao_coluna  valor_inserido  presente_fujita  presente_mathys
AL627309.5 ENSG00000241860               6             0.5             True            False
AP006222.2 ENSG00000286448               8             0.5             True            False
    OR4F29 ENSG00000284733              10             0.5             True            False
    OR4F16 ENSG00000284662              12             0.5             True            False
 LINC01128 ENSG00000228794              16             0.5             True            False
AL645608.5 ENSG00000242590              33             0.5             True            False
AL390719.3 ENSG00000285812              37             0.5             True            False
AL139287.1 ENSG00000240731              54             0.5             True            False
AL691432.4 ENSG00000286989              76             0.5             Tru

## Visualização — 3 primeiras linhas do Mathys alinhado

Confirma que os genes adicionados têm valor **0.5** e os genes presentes têm seus valores reais de expressão.

In [47]:
import anndata as ad
import scipy.sparse as sp
import numpy as np

print("Carregando 3 primeiras células do Mathys alinhado (backed='r')...")
_m = ad.read_h5ad(out_m, backed='r')

# Extrai as 3 primeiras células
X_raw = _m[:3].X
X_3   = X_raw.toarray() if sp.issparse(X_raw) else np.asarray(X_raw)
obs_nomes    = _m.obs_names[:3].tolist()
var_nomes    = _m.var_names.tolist()
_m.file.close()
del _m, X_raw

# ── Amostra de genes ADICIONADOS (devem mostrar 0.5) ──────────────────────────
added_idx   = df_tracking['posicao_coluna'].values[:6]
added_genes = df_tracking['ensembl_id'].values[:6]
added_names = df_tracking['gene_name'].values[:6]

df_added = pd.DataFrame(
    X_3[:, added_idx],
    index=obs_nomes,
    columns=[f"{n}\n({e})" for n, e in zip(added_names, added_genes)],
)

# ── Amostra de genes PRESENTES no Mathys (valores reais de expressão) ─────────
added_set     = set(df_tracking['posicao_coluna'].tolist())
present_idx   = [i for i in range(len(var_nomes)) if i not in added_set][:6]
present_genes = [var_nomes[i] for i in present_idx]

df_present = pd.DataFrame(
    X_3[:, present_idx],
    index=obs_nomes,
    columns=present_genes,
)

print("\n── 3 primeiras células · genes ADICIONADOS com 0.5 (amostra de 6) ──")
print(df_added.to_string())

print("\n── 3 primeiras células · genes PRESENTES no Mathys (amostra de 6) ──")
print(df_present.to_string())

del X_3

Carregando 3 primeiras células do Mathys alinhado (backed='r')...

── 3 primeiras células · genes ADICIONADOS com 0.5 (amostra de 6) ──
                    AL627309.5\n(ENSG00000241860)  AP006222.2\n(ENSG00000286448)  OR4F29\n(ENSG00000284733)  OR4F16\n(ENSG00000284662)  LINC01128\n(ENSG00000228794)  AL645608.5\n(ENSG00000242590)
AAAGATGCACGGTGTC-1                            0.5                            0.5                        0.5                        0.5                           0.5                            0.5
AAATGCCTCCAATGGT-1                            0.5                            0.5                        0.5                        0.5                           0.5                            0.5
AACCATGTCTGTACGA-1                            0.5                            0.5                        0.5                        0.5                           0.5                            0.5

── 3 primeiras células · genes PRESENTES no Mathys (amostra de 6) ──
          

In [48]:
# Os arquivos já foram salvos dentro do Passo 3.
# Esta célula verifica que existem no disco.
import os

OUT_DIR = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Testes Hopifild"
for fname in ["adataF_ensembl_alinhado.h5ad", "adataM_ensembl_alinhado.h5ad"]:
    path = os.path.join(OUT_DIR, fname)
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else None
    status = f"{size_mb:.0f} MB" if size_mb else "NÃO ENCONTRADO"
    print(f"  {fname}: {status}")

  adataF_ensembl_alinhado.h5ad: 1186 MB
  adataM_ensembl_alinhado.h5ad: 3301 MB


## Passo 4 — Validação da ordem de genes

Garante que **ambos os AnnData alinhados possuem exatamente a mesma lista de genes,
na mesma ordem**, e que essa ordem é idêntica à ordem original do Fujita (`genes_ordenados`).

Essa verificação é obrigatória porque a `rede35` foi treinada com os índices de coluna
do Fujita: trocar ou reordenar genes invalidaria todas as previsões da rede.

Se qualquer divergência for encontrada, uma exceção é lançada **antes** de qualquer uso
posterior dos arquivos.

In [49]:
import anndata as ad
import numpy as np

# ── Carregar apenas os metadados (backed='r') para não desperdiçar RAM ───────
print("Carregando metadados dos arquivos alinhados...")
_f = ad.read_h5ad(out_f, backed='r')
_m = ad.read_h5ad(out_m, backed='r')

genes_f = _f.var_names.tolist()
genes_m = _m.var_names.tolist()

_f.file.close()
_m.file.close()
del _f, _m

# ── 1. Número de genes ────────────────────────────────────────────────────────
if len(genes_f) != len(genes_ordenados):
    raise ValueError(
        f"[VALIDAÇÃO FALHOU] Fujita alinhado tem {len(genes_f)} genes, "
        f"mas genes_ordenados tem {len(genes_ordenados)}. "
        "O arquivo foi salvo corretamente?"
    )

if len(genes_m) != len(genes_ordenados):
    raise ValueError(
        f"[VALIDAÇÃO FALHOU] Mathys alinhado tem {len(genes_m)} genes, "
        f"mas genes_ordenados tem {len(genes_ordenados)}. "
        "O arquivo foi salvo corretamente?"
    )

# ── 2. Ordem idêntica à referência Fujita (genes_ordenados) ──────────────────
divergencias_f = [
    (i, genes_ordenados[i], genes_f[i])
    for i in range(len(genes_ordenados))
    if genes_f[i] != genes_ordenados[i]
]
if divergencias_f:
    primeiros = divergencias_f[:5]
    msg = "\n  ".join(
        f"pos {i}: esperado={esperado!r}  encontrado={encontrado!r}"
        for i, esperado, encontrado in primeiros
    )
    raise ValueError(
        f"[VALIDAÇÃO FALHOU] Fujita alinhado diverge da ordem original do Fujita "
        f"em {len(divergencias_f)} posição(ões). Primeiras divergências:\n  {msg}"
    )

divergencias_m = [
    (i, genes_ordenados[i], genes_m[i])
    for i in range(len(genes_ordenados))
    if genes_m[i] != genes_ordenados[i]
]
if divergencias_m:
    primeiros = divergencias_m[:5]
    msg = "\n  ".join(
        f"pos {i}: esperado={esperado!r}  encontrado={encontrado!r}"
        for i, esperado, encontrado in primeiros
    )
    raise ValueError(
        f"[VALIDAÇÃO FALHOU] Mathys alinhado diverge da ordem original do Fujita "
        f"em {len(divergencias_m)} posição(ões). Primeiras divergências:\n  {msg}"
    )

# ── 3. Fujita e Mathys idênticos entre si ────────────────────────────────────
divergencias_fm = [
    (i, genes_f[i], genes_m[i])
    for i in range(len(genes_f))
    if genes_f[i] != genes_m[i]
]
if divergencias_fm:
    primeiros = divergencias_fm[:5]
    msg = "\n  ".join(
        f"pos {i}: Fujita={gf!r}  Mathys={gm!r}"
        for i, gf, gm in primeiros
    )
    raise ValueError(
        f"[VALIDAÇÃO FALHOU] Fujita e Mathys alinhados divergem entre si "
        f"em {len(divergencias_fm)} posição(ões). Primeiras divergências:\n  {msg}"
    )

# ── Tudo OK ───────────────────────────────────────────────────────────────────
print(f"✓ Número de genes idêntico: {len(genes_ordenados)}")
print(f"✓ Fujita alinhado == ordem original Fujita")
print(f"✓ Mathys alinhado == ordem original Fujita")
print(f"✓ Fujita alinhado == Mathys alinhado")
print("\nValidação concluída com sucesso. Os arquivos estão prontos para a rede35.")


Carregando metadados dos arquivos alinhados...
✓ Número de genes idêntico: 36601
✓ Fujita alinhado == ordem original Fujita
✓ Mathys alinhado == ordem original Fujita
✓ Fujita alinhado == Mathys alinhado

Validação concluída com sucesso. Os arquivos estão prontos para a rede35.


## Análise — Cobertura dos Top 5000 genes frequentes do Fujita no Mathys

Verifica quantos dos 5000 genes mais frequentes do Fujita estão realmente expressos no Mathys
(i.e., têm Ensembl ID presente no espaço gênico original do Mathys, não apenas como coluna zerada).

In [51]:
import pandas as pd

TOP5000_PATH = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Testes Hopifild/top_5000_frequentes.csv"

top5000 = pd.read_csv(TOP5000_PATH)
print(f"Genes no arquivo top_5000: {len(top5000)}")
print(top5000.head(3))

# map_f e map_m já estão definidos nas células anteriores
# map_f: {gene_name -> ensembl_id} do Fujita
# ids_m: conjunto de Ensembl IDs presentes no Mathys original (sem zeros)

ids_m_set = set(map_m.values())

resultados = []
for _, row in top5000.iterrows():
    gene_name = row['gene']
    frequencia = row['frequencia']
    ensembl_id = map_f.get(gene_name, None)          # converte nome → Ensembl
    no_mathys = (ensembl_id in ids_m_set) if ensembl_id else False
    sem_ensembl = ensembl_id is None
    resultados.append({
        'gene_name': gene_name,
        'frequencia': frequencia,
        'ensembl_id': ensembl_id if ensembl_id else 'N/A',
        'presente_mathys': no_mathys,
        'sem_ensembl_fujita': sem_ensembl,
    })

df_res = pd.DataFrame(resultados)

total        = len(df_res)
presentes    = df_res['presente_mathys'].sum()
ausentes     = total - presentes - df_res['sem_ensembl_fujita'].sum()
sem_ensembl  = df_res['sem_ensembl_fujita'].sum()

print(f"\n{'='*50}")
print(f"  Top {total} genes frequentes do Fujita")
print(f"{'='*50}")
print(f"  Presentes no Mathys       : {presentes:>5}  ({presentes/total*100:.1f}%)")
print(f"  Ausentes no Mathys (zeros): {total - presentes - sem_ensembl:>5}  ({(total - presentes - sem_ensembl)/total*100:.1f}%)")
print(f"  Sem Ensembl ID no Fujita  : {sem_ensembl:>5}  ({sem_ensembl/total*100:.1f}%)")
print(f"{'='*50}")

print("\n-- 10 genes presentes no Mathys (maior frequência) --")
print(df_res[df_res['presente_mathys']].head(10)[['gene_name','frequencia','ensembl_id']].to_string(index=False))

print("\n-- 10 genes AUSENTES no Mathys (maior frequência) --")
ausentes_df = df_res[~df_res['presente_mathys'] & ~df_res['sem_ensembl_fujita']]
print(ausentes_df.head(10)[['gene_name','frequencia','ensembl_id']].to_string(index=False))

# Salva o resultado completo
out_csv = r"/home/rasert/Leticia Delicia/Sweep-Harmonization/Meus_testes/Testes Hopifild/top5000_cobertura_mathys.csv"
df_res.to_csv(out_csv, index=False)
print(f"\nResultado completo salvo em: {out_csv}")

Genes no arquivo top_5000: 5000
     gene  frequencia
0  MALAT1       40782
1   CADM2       39065
2   PCDH9       39012

  Top 5000 genes frequentes do Fujita
  Presentes no Mathys       :  4906  (98.1%)
  Ausentes no Mathys (zeros):    94  (1.9%)
  Sem Ensembl ID no Fujita  :     0  (0.0%)

-- 10 genes presentes no Mathys (maior frequência) --
gene_name  frequencia      ensembl_id
   MALAT1       40782 ENSG00000251562
    CADM2       39065 ENSG00000175161
    PCDH9       39012 ENSG00000184226
     DLG2       38107 ENSG00000150672
   MT-CO3       37612 ENSG00000198938
   MT-ND3       37498 ENSG00000198840
     ANK2       37429 ENSG00000145362
    LSAMP       37397 ENSG00000185565
    MAGI2       37390 ENSG00000187391
   ADGRB3       37355 ENSG00000135298

-- 10 genes AUSENTES no Mathys (maior frequência) --
  gene_name  frequencia      ensembl_id
     SNHG14       38439 ENSG00000224078
      GTF2I       32786 ENSG00000263001
   MIR100HG       29839 ENSG00000255248
IQCJ-SCHIP1       280